# Multi-Model Fine-Tuning with LoRA
## Content Safety Dataset - Comparative Analysis

This notebook fine-tunes 3 different models using LoRA on the Nvidia Aegis AI Content Safety Dataset:
- Qwen3-0.6B-Base
- Llama-3.2-1B
- ERNIE-4.5-0.3B-Base-PT

## 1. Environment Setup

In [ ]:
# Install required packages - Fix bitsandbytes installation
import sys
import subprocess

# First, ensure we have the latest pip
!pip install --upgrade pip

# Install critical dependencies first
print("Installing sentencepiece and protobuf...")
!pip install sentencepiece>=0.1.99 protobuf>=3.20.0

# Install PyTorch with CUDA support (adjust based on your CUDA version)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install bitsandbytes from source or pre-built wheel
# Try multiple approaches for bitsandbytes
print("\nInstalling bitsandbytes...")
try:
    !pip uninstall -y bitsandbytes
    !pip install bitsandbytes>=0.43.0
    print("✓ bitsandbytes installed successfully")
except:
    print("⚠ Standard bitsandbytes install failed, trying alternative...")
    try:
        !pip install bitsandbytes --no-cache-dir --force-reinstall
        print("✓ bitsandbytes installed with force reinstall")
    except:
        print("⚠ bitsandbytes installation failed - will use FP16 training")

# Install other required packages
print("\nInstalling transformers ecosystem...")
!pip install -q transformers>=4.38.0 datasets>=2.16.0 accelerate>=0.26.0 peft>=0.8.0 trl>=0.7.10
!pip install -q huggingface_hub
!pip install -q scikit-learn pandas numpy matplotlib seaborn

print("\n" + "="*60)
print("All packages installed successfully!")
print("="*60)

# Verify installations
print("\nVerifying key packages...")
try:
    import sentencepiece
    print(f"✓ sentencepiece version: {sentencepiece.__version__}")
except:
    print("✗ sentencepiece not found")

try:
    import google.protobuf
    print(f"✓ protobuf version: {google.protobuf.__version__}")
except:
    print("✗ protobuf not found")

try:
    import transformers
    print(f"✓ transformers version: {transformers.__version__}")
except:
    print("✗ transformers not found")

try:
    import peft
    print(f"✓ peft version: {peft.__version__}")
except:
    print("✗ peft not found")

try:
    import bitsandbytes
    print(f"✓ bitsandbytes version: {bitsandbytes.__version__}")
except:
    print("⚠ bitsandbytes not available (will use FP16)")

print("\n" + "="*60)
print("Setup verification complete!")
print("="*60)

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login, HfApi
import json
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Set up plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Login to HuggingFace Hub
# You'll need to enter your HF token
login()

## 2. Configuration

In [ ]:
# Model configurations - Only 3 models
# Models will be uploaded to https://huggingface.co/ahczhg
MODELS = {
    "qwen3-0.6b": {
        "model_id": "Qwen/Qwen3-0.6B-Base",
        "hub_name": "ahczhg/qwen3-0.6b-aegis-safety-lora"
    },
    "llama-3.2-1b": {
        "model_id": "meta-llama/Llama-3.2-1B",
        "hub_name": "ahczhg/llama-3.2-1b-aegis-safety-lora"
    },
    "ernie-0.3b": {
        "model_id": "baidu/ERNIE-4.5-0.3B-Base-PT",
        "hub_name": "ahczhg/ernie-4.5-0.3b-aegis-safety-lora"
    }
}

# Dataset configuration - Using 2000 samples
DATASET_NAME = "nvidia/Aegis-AI-Content-Safety-Dataset-2.0"
NUM_SAMPLES = 2000

# LoRA configuration
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
}

# Training configuration
TRAINING_CONFIG = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "fp16": True,
    "optim": "paged_adamw_8bit",
    "max_grad_norm": 0.3,
    "max_seq_length": 512
}

# Check if bitsandbytes is available and configure quantization
USE_QUANTIZATION = True
BNB_CONFIG = None

try:
    from transformers import BitsAndBytesConfig
    import bitsandbytes as bnb
    
    # Quantization config for loading large models
    BNB_CONFIG = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    print("✓ Quantization (4-bit) enabled with bitsandbytes")
except Exception as e:
    print(f"⚠ Warning: Could not initialize bitsandbytes quantization: {e}")
    print("⚠ Will attempt to use full precision or fp16. This requires more GPU memory.")
    USE_QUANTIZATION = False
    
    # Fallback: adjust batch size and other settings for non-quantized training
    TRAINING_CONFIG["per_device_train_batch_size"] = 1
    TRAINING_CONFIG["gradient_accumulation_steps"] = 16
    print("⚠ Adjusted batch size to 1 with gradient accumulation 16 to fit in memory")

# Results storage
results = {}

print("\n" + "="*60)
print("Configuration loaded successfully!")
print("="*60)
print(f"HuggingFace Account: ahczhg")
print(f"Number of models: {len(MODELS)}")
print(f"Dataset samples: {NUM_SAMPLES}")
print(f"Quantization enabled: {USE_QUANTIZATION}")
print(f"Batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"Effective batch size: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print("\nModels will be uploaded to:")
for key, config in MODELS.items():
    print(f"  - {config['hub_name']}")
print("="*60)

## 3. Load and Prepare Dataset

In [ ]:
# Load the dataset
print(f"Loading {DATASET_NAME}...")
dataset = load_dataset(DATASET_NAME, split="train")

# Select 10,000 samples
if len(dataset) > NUM_SAMPLES:
    dataset = dataset.shuffle(seed=42).select(range(NUM_SAMPLES))
else:
    print(f"Warning: Dataset has only {len(dataset)} samples, using all available data")

print(f"Dataset loaded: {len(dataset)} samples")
print(f"\nDataset features: {dataset.features}")
print(f"\nFirst example:")
print(dataset[0])

In [ ]:
# Split dataset into train/validation/test
train_test = dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_dataset = train_test['train']
valid_dataset = test_valid['train']
test_dataset = test_valid['test']

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(valid_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Format the dataset for training
def format_instruction(example):
    """Format the dataset into instruction-following format"""
    # Adjust this based on actual dataset structure
    # This is a template - you may need to modify based on actual fields
    if 'prompt' in example and 'response' in example:
        text = f"### Instruction:\n{example['prompt']}\n\n### Response:\n{example['response']}"
    elif 'text' in example:
        text = example['text']
    elif 'input' in example and 'output' in example:
        text = f"### Input:\n{example['input']}\n\n### Output:\n{example['output']}"
    else:
        # Fallback - use first text field found
        text = str(example[list(example.keys())[0]])
    
    return {"text": text}

# Apply formatting
train_dataset = train_dataset.map(format_instruction, remove_columns=train_dataset.column_names)
valid_dataset = valid_dataset.map(format_instruction, remove_columns=valid_dataset.column_names)
test_dataset = test_dataset.map(format_instruction, remove_columns=test_dataset.column_names)

print("Dataset formatted successfully!")
print(f"\nFormatted example:\n{train_dataset[0]['text'][:500]}...")

## 4. Helper Functions

In [ ]:
def generate_model_card(model_key, model_config, training_results=None):
    """Generate a comprehensive README.md for the model"""
    
    model_descriptions = {
        "qwen3-0.6b": {
            "name": "Qwen3-0.6B Fine-tuned on Aegis AI Content Safety",
            "base_model": "Qwen/Qwen3-0.6B-Base",
            "description": "This is a fine-tuned version of Qwen3-0.6B-Base, optimized for content safety classification and moderation tasks. Qwen3 is a compact yet powerful language model developed by Alibaba Cloud, designed for efficient deployment while maintaining strong performance.",
            "capabilities": [
                "Content safety classification",
                "Toxic content detection",
                "Harmful content identification",
                "Safety-aware text generation",
                "Content moderation assistance"
            ],
            "use_cases": [
                "Content moderation systems",
                "Chat application safety filters",
                "User-generated content screening",
                "Educational content filtering",
                "Social media safety monitoring"
            ]
        },
        "llama-3.2-1b": {
            "name": "Llama-3.2-1B Fine-tuned on Aegis AI Content Safety",
            "base_model": "meta-llama/Llama-3.2-1B",
            "description": "This model is a fine-tuned version of Meta's Llama-3.2-1B, specialized for content safety and moderation tasks. Llama-3.2 represents Meta's latest advancement in efficient language models, offering excellent performance in a compact 1B parameter size.",
            "capabilities": [
                "Advanced content safety analysis",
                "Multi-category threat detection",
                "Context-aware safety classification",
                "Nuanced harmful content identification",
                "Safety-first response generation"
            ],
            "use_cases": [
                "Enterprise content moderation",
                "Real-time chat safety monitoring",
                "Content policy enforcement",
                "Community guidelines compliance",
                "Automated content review systems"
            ]
        },
        "ernie-0.3b": {
            "name": "ERNIE-4.5-0.3B Fine-tuned on Aegis AI Content Safety",
            "base_model": "baidu/ERNIE-4.5-0.3B-Base-PT",
            "description": "This is a fine-tuned version of Baidu's ERNIE-4.5-0.3B-Base, optimized for content safety applications. ERNIE (Enhanced Representation through kNowledge IntEgration) is particularly strong in understanding context and knowledge-enhanced language understanding, making it ideal for nuanced safety detection.",
            "capabilities": [
                "Context-aware safety detection",
                "Knowledge-enhanced content analysis",
                "Cultural and linguistic sensitivity",
                "Fine-grained threat classification",
                "Efficient content safety screening"
            ],
            "use_cases": [
                "Cross-cultural content moderation",
                "Educational platform safety",
                "Lightweight mobile applications",
                "Edge device content filtering",
                "Multi-lingual safety systems"
            ]
        }
    }
    
    info = model_descriptions.get(model_key, {})
    base_model = info.get("base_model", model_config["model_id"])
    
    # Training info
    training_info = ""
    if training_results:
        training_time = training_results.get('training_time', 0) / 60
        perplexity = training_results.get('eval_results', {}).get('avg_perplexity', 'N/A')
        training_info = f"""
## Training Results

- **Training Time**: {training_time:.2f} minutes
- **Average Perplexity**: {perplexity if isinstance(perplexity, str) else f'{perplexity:.2f}'}
- **Dataset Size**: 2,000 samples
- **Training Epochs**: 3
"""
    
    readme = f"""---
language:
- en
license: apache-2.0
base_model: {base_model}
tags:
- content-safety
- content-moderation
- safety
- lora
- fine-tuned
- nvidia-aegis
- text-classification
datasets:
- nvidia/Aegis-AI-Content-Safety-Dataset-2.0
metrics:
- perplexity
- accuracy
library_name: transformers
pipeline_tag: text-classification
---

# {info.get('name', 'Content Safety Fine-tuned Model')}

## Model Description

{info.get('description', 'This model has been fine-tuned for content safety tasks.')}

This model was fine-tuned using **LoRA (Low-Rank Adaptation)** on the [NVIDIA Aegis AI Content Safety Dataset 2.0](https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0), which contains diverse examples of safe and unsafe content across multiple categories.

## Model Details

- **Base Model**: [{base_model}](https://huggingface.co/{base_model})
- **Fine-tuning Method**: LoRA (Low-Rank Adaptation)
- **Dataset**: NVIDIA Aegis AI Content Safety Dataset 2.0
- **Training Samples**: 2,000 carefully selected samples
- **Language**: English
- **License**: Apache 2.0

## Capabilities

{chr(10).join([f'- {cap}' for cap in info.get('capabilities', [])])}

## Intended Use Cases

{chr(10).join([f'- {use}' for use in info.get('use_cases', [])])}
{training_info}
## Training Configuration

### LoRA Parameters
- **Rank (r)**: 16
- **Alpha**: 32
- **Dropout**: 0.05
- **Target Modules**: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj

### Training Hyperparameters
- **Learning Rate**: 2e-4
- **Batch Size**: 4 (per device)
- **Gradient Accumulation Steps**: 4
- **Effective Batch Size**: 16
- **Epochs**: 3
- **Optimizer**: AdamW (8-bit paged)
- **LR Scheduler**: Cosine with warmup
- **Warmup Ratio**: 0.1
- **FP16 Training**: Yes
- **Max Sequence Length**: 512

## Usage

### Installation

```bash
pip install transformers torch peft
```

### Basic Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model and tokenizer
model_name = "{model_config['hub_name']}"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Example: Content safety check
prompt = "### Instruction:\\nAnalyze this content for safety: 'Your text here'\\n\\n### Response:\\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
        top_p=0.95
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)
```

### Advanced Usage with Pipeline

```python
from transformers import pipeline

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model="{model_config['hub_name']}",
    torch_dtype=torch.float16,
    device_map="auto"
)

# Generate safety analysis
result = generator(
    "### Instruction:\\nIs this content safe? 'Hello, how are you?'\\n\\n### Response:\\n",
    max_new_tokens=128,
    temperature=0.7,
    do_sample=True
)

print(result[0]['generated_text'])
```

## Evaluation

The model was evaluated on a held-out test set from the Aegis AI Content Safety Dataset. Key metrics include:

- Perplexity on validation set
- Content safety classification accuracy
- False positive/negative rates for harmful content detection

## Limitations

- The model is primarily trained on English language content
- Performance may vary on domain-specific or highly technical content
- Should be used as part of a comprehensive content moderation system, not as the sole decision-maker
- May require fine-tuning for specific use cases or content domains
- The model's outputs should be reviewed by human moderators for critical applications

## Ethical Considerations

- This model is designed to assist in content safety and moderation tasks
- It should not be used to censor legitimate speech or suppress diverse viewpoints
- Decisions about content moderation should involve human oversight
- The model may reflect biases present in the training data
- Users should implement appropriate safeguards and appeal processes

## Training Data

The model was fine-tuned on the [NVIDIA Aegis AI Content Safety Dataset 2.0](https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0), which includes:

- Diverse examples of safe and unsafe content
- Multiple categories of potentially harmful content
- Balanced representation of safe content
- Real-world scenarios and edge cases

## Citation

If you use this model in your research or applications, please cite:

```bibtex
@misc{{{model_key.replace('-', '_')}_aegis_safety,
  author = {{ahczhg}},
  title = {{{info.get('name', 'Content Safety Fine-tuned Model')}}},
  year = {{2025}},
  publisher = {{HuggingFace}},
  howpublished = {{\\url{{https://huggingface.co/{model_config['hub_name']}}}}},
  note = {{Fine-tuned on NVIDIA Aegis AI Content Safety Dataset 2.0}}
}}
```

## Acknowledgments

- Base model by the original authors: {base_model.split('/')[0]}
- Dataset provided by NVIDIA
- Fine-tuning performed using HuggingFace Transformers and PEFT libraries

## Contact

For questions, issues, or feedback, please visit the [model repository](https://huggingface.co/{model_config['hub_name']}).

## Model Card Authors

- ahczhg

## Model Card Contact

- https://huggingface.co/ahczhg
"""
    
    return readme


def load_model_and_tokenizer(model_id, use_quantization=USE_QUANTIZATION):
    """Load model and tokenizer with optional quantization"""
    print(f"\nLoading {model_id}...")
    print(f"Quantization: {'Enabled (4-bit)' if use_quantization and BNB_CONFIG else 'Disabled (FP16)'}")
    
    # Load tokenizer with better error handling
    print("Loading tokenizer...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            padding_side="right"
        )
    except (ImportError, ValueError) as e:
        print(f"⚠ Error loading tokenizer: {e}")
        print("⚠ Attempting to load tokenizer with use_fast=False...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                model_id,
                trust_remote_code=True,
                padding_side="right",
                use_fast=False  # Use slow tokenizer as fallback
            )
            print("✓ Loaded slow tokenizer successfully")
        except Exception as e2:
            print(f"✗ Failed to load tokenizer: {e2}")
            raise RuntimeError(
                "Failed to load tokenizer. Please ensure sentencepiece and protobuf are installed:\n"
                "  pip install sentencepiece protobuf\n"
                "Then restart the kernel and try again."
            )
    
    # Set pad token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    print("✓ Tokenizer loaded successfully")
    
    # Load model with appropriate configuration
    print("Loading model...")
    load_kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
        "torch_dtype": torch.float16
    }
    
    # Only add quantization config if bitsandbytes is available
    if use_quantization and BNB_CONFIG is not None:
        load_kwargs["quantization_config"] = BNB_CONFIG
    
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    except Exception as e:
        print(f"⚠ Error loading with current settings: {e}")
        print("⚠ Retrying without quantization...")
        # Remove quantization and try again
        load_kwargs.pop("quantization_config", None)
        model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    
    # Prepare model for training
    if use_quantization and BNB_CONFIG is not None:
        try:
            from peft import prepare_model_for_kbit_training
            model = prepare_model_for_kbit_training(model)
            print("✓ Model prepared for k-bit training")
        except Exception as e:
            print(f"⚠ Could not prepare for k-bit training: {e}")
    
    model.config.use_cache = False
    if hasattr(model.config, 'pretraining_tp'):
        model.config.pretraining_tp = 1
    
    # Enable gradient checkpointing to save memory
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
        print("✓ Gradient checkpointing enabled")
    
    print(f"✓ Model and tokenizer loaded successfully!")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model size: {total_params / 1e9:.2f}B parameters")
    
    return model, tokenizer


def setup_lora(model):
    """Setup LoRA adapters"""
    lora_config = LoraConfig(**LORA_CONFIG)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


def train_model(model, tokenizer, model_name, output_dir):
    """Train a model with LoRA"""
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}\n")
    
    # Setup LoRA
    model = setup_lora(model)
    
    # Determine optimizer based on quantization
    optimizer = TRAINING_CONFIG["optim"]
    if not USE_QUANTIZATION or BNB_CONFIG is None:
        # Use standard AdamW if bitsandbytes is not available
        optimizer = "adamw_torch"
        print("⚠ Using standard AdamW optimizer (bitsandbytes not available)")
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
        warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
        logging_steps=TRAINING_CONFIG["logging_steps"],
        save_strategy=TRAINING_CONFIG["save_strategy"],
        fp16=TRAINING_CONFIG["fp16"],
        optim=optimizer,
        max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
        eval_strategy="epoch",
        report_to="none",  # Change to "wandb" if you want to use W&B
        gradient_checkpointing=True,
        save_total_limit=2,  # Keep only last 2 checkpoints to save disk space
    )
    
    # Prepare formatting function for SFTTrainer
    def formatting_func(example):
        return example["text"]
    
    # Initialize trainer with correct API
    try:
        # Try newer API (TRL >= 0.7.0)
        trainer = SFTTrainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=valid_dataset,
            processing_class=tokenizer,  # New API uses processing_class
            args=training_args,
            dataset_text_field="text",
            max_seq_length=TRAINING_CONFIG["max_seq_length"],
            packing=False
        )
    except TypeError:
        try:
            # Try alternative API
            trainer = SFTTrainer(
                model=model,
                train_dataset=train_dataset,
                eval_dataset=valid_dataset,
                formatting_func=formatting_func,
                args=training_args,
                max_seq_length=TRAINING_CONFIG["max_seq_length"],
                packing=False
            )
            # Set tokenizer separately
            trainer.tokenizer = tokenizer
        except TypeError:
            # Fallback to basic Trainer
            print("⚠ Using basic Trainer instead of SFTTrainer")
            from transformers import Trainer, DataCollatorForLanguageModeling
            
            # Tokenize datasets
            def tokenize_function(examples):
                return tokenizer(examples["text"], truncation=True, max_length=TRAINING_CONFIG["max_seq_length"])
            
            tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
            tokenized_eval = valid_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
            
            data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
            
            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_train,
                eval_dataset=tokenized_eval,
                data_collator=data_collator,
            )
    
    # Train
    start_time = datetime.now()
    print(f"Training started at {start_time}")
    print(f"Effective batch size: {TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
    print(f"Total training steps: {len(train_dataset) // (TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']) * TRAINING_CONFIG['num_train_epochs']}")
    
    trainer.train()
    
    end_time = datetime.now()
    training_time = (end_time - start_time).total_seconds()
    print(f"\n✓ Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
    
    # Save the model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✓ Model saved to {output_dir}")
    
    # Get training metrics
    metrics = trainer.state.log_history
    
    return model, tokenizer, metrics, training_time


def evaluate_model(model, tokenizer, model_name):
    """Evaluate model on test set"""
    print(f"\nEvaluating {model_name}...")
    
    model.eval()
    
    # Create text generation pipeline
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.95
    )
    
    # Evaluate on a subset of test data
    test_samples = test_dataset.select(range(min(100, len(test_dataset))))
    
    predictions = []
    prompts = []
    
    for i, example in enumerate(test_samples):
        if i % 20 == 0:
            print(f"Processing sample {i+1}/{len(test_samples)}...")
        
        prompt = example['text'].split('### Response:')[0] if '### Response:' in example['text'] else example['text'][:100]
        prompts.append(prompt)
        
        try:
            output = generator(prompt, max_new_tokens=128)[0]['generated_text']
            predictions.append(output)
        except Exception as e:
            print(f"⚠ Error generating for sample {i}: {e}")
            predictions.append("")
    
    # Calculate perplexity on validation set
    print("Calculating perplexity...")
    perplexities = []
    
    with torch.no_grad():
        for i, example in enumerate(valid_dataset.select(range(min(50, len(valid_dataset))))):
            if i % 10 == 0:
                print(f"Perplexity calculation: {i+1}/50...")
            
            inputs = tokenizer(example['text'], return_tensors='pt', truncation=True, max_length=512)
            
            # Move inputs to same device as model
            if hasattr(model, 'device'):
                inputs = {k: v.to(model.device) for k, v in inputs.items()}
            else:
                inputs = {k: v.to('cuda' if torch.cuda.is_available() else 'cpu') for k, v in inputs.items()}
            
            try:
                outputs = model(**inputs, labels=inputs['input_ids'])
                perplexity = torch.exp(outputs.loss).item()
                if not np.isnan(perplexity) and not np.isinf(perplexity):
                    perplexities.append(perplexity)
            except Exception as e:
                print(f"⚠ Error calculating perplexity for sample {i}: {e}")
                continue
    
    avg_perplexity = np.mean(perplexities) if perplexities else float('inf')
    
    eval_results = {
        'avg_perplexity': avg_perplexity,
        'num_test_samples': len(test_samples),
        'sample_predictions': predictions[:5],
        'sample_prompts': prompts[:5]
    }
    
    print(f"✓ Average Perplexity: {avg_perplexity:.2f}")
    
    return eval_results


def upload_to_hub(model_path, hub_name, model_name, model_key):
    """Upload model to HuggingFace Hub with README"""
    print(f"\nUploading {model_name} to HuggingFace Hub as {hub_name}...")
    
    try:
        from peft import PeftModel, PeftConfig
        
        # Get training results if available
        training_results = results.get(model_key, {})
        
        # Generate README.md
        print("Generating README.md...")
        readme_content = generate_model_card(
            model_key, 
            MODELS[model_key],
            training_results
        )
        
        # Save README locally
        readme_path = os.path.join(model_path, "README.md")
        with open(readme_path, 'w', encoding='utf-8') as f:
            f.write(readme_content)
        print(f"✓ README.md saved to {readme_path}")
        
        # For LoRA models, we need to merge and upload
        print("Loading LoRA adapter...")
        config = PeftConfig.from_pretrained(model_path)
        
        # Load base model
        base_model = AutoModelForCausalLM.from_pretrained(
            config.base_model_name_or_path,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16
        )
        
        # Load LoRA adapter
        model = PeftModel.from_pretrained(base_model, model_path)
        
        # Merge LoRA weights with base model
        print("Merging LoRA weights with base model...")
        model = model.merge_and_unload()
        
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        
        # Push to hub
        print(f"Pushing model to hub: {hub_name}")
        model.push_to_hub(hub_name, private=False)
        
        print(f"Pushing tokenizer to hub: {hub_name}")
        tokenizer.push_to_hub(hub_name, private=False)
        
        # Upload README separately to ensure it's included
        print(f"Uploading README.md...")
        from huggingface_hub import upload_file
        upload_file(
            path_or_fileobj=readme_path,
            path_in_repo="README.md",
            repo_id=hub_name,
            repo_type="model"
        )
        
        print(f"✓ Successfully uploaded to https://huggingface.co/{hub_name}")
        
        # Cleanup
        del model, base_model, tokenizer
        torch.cuda.empty_cache()
        
        return True
    except Exception as e:
        print(f"✗ Error uploading {model_name}: {e}")
        import traceback
        traceback.print_exc()
        return False

print("\n" + "="*60)
print("Helper functions defined successfully!")
print("="*60)

## 5. Fine-tune Models

### 5.1 Qwen3-0.6B-Base

In [ ]:
model_key = "qwen3-0.6b"
model_id = MODELS[model_key]["model_id"]
output_dir = f"./output/{model_key}"

# Load model
model, tokenizer = load_model_and_tokenizer(model_id)

# Train
model, tokenizer, metrics, training_time = train_model(model, tokenizer, model_key, output_dir)

# Evaluate
eval_results = evaluate_model(model, tokenizer, model_key)

# Store results
results[model_key] = {
    'training_time': training_time,
    'eval_results': eval_results,
    'metrics': metrics,
    'output_dir': output_dir
}

# Clear memory
del model, tokenizer
torch.cuda.empty_cache()

print(f"\n✓ {model_key} training completed!")

### 5.2 Llama-3.2-1B

In [ ]:
model_key = "llama-3.2-1b"
model_id = MODELS[model_key]["model_id"]
output_dir = f"./output/{model_key}"

# Load model
model, tokenizer = load_model_and_tokenizer(model_id)

# Train
model, tokenizer, metrics, training_time = train_model(model, tokenizer, model_key, output_dir)

# Evaluate
eval_results = evaluate_model(model, tokenizer, model_key)

# Store results
results[model_key] = {
    'training_time': training_time,
    'eval_results': eval_results,
    'metrics': metrics,
    'output_dir': output_dir
}

# Clear memory
del model, tokenizer
torch.cuda.empty_cache()

print(f"\n✓ {model_key} training completed!")

### 5.3 ERNIE-4.5-0.3B-Base-PT

In [ ]:
model_key = "ernie-0.3b"
model_id = MODELS[model_key]["model_id"]
output_dir = f"./output/{model_key}"

# Load model
model, tokenizer = load_model_and_tokenizer(model_id)

# Train
model, tokenizer, metrics, training_time = train_model(model, tokenizer, model_key, output_dir)

# Evaluate
eval_results = evaluate_model(model, tokenizer, model_key)

# Store results
results[model_key] = {
    'training_time': training_time,
    'eval_results': eval_results,
    'metrics': metrics,
    'output_dir': output_dir
}

# Clear memory
del model, tokenizer
torch.cuda.empty_cache()

print(f"\n✓ {model_key} training completed!")

In [ ]:
model_key = "ernie-0.3b"
model_id = MODELS[model_key]["model_id"]
output_dir = f"./output/{model_key}"

# Load model
model, tokenizer = load_model_and_tokenizer(model_id)

# Train
model, tokenizer, metrics, training_time = train_model(model, tokenizer, model_key, output_dir)

# Evaluate
eval_results = evaluate_model(model, tokenizer, model_key)

# Store results
results[model_key] = {
    'training_time': training_time,
    'eval_results': eval_results,
    'metrics': metrics,
    'output_dir': output_dir
}

# Clear memory
del model, tokenizer
torch.cuda.empty_cache()

print(f"\n✓ {model_key} training completed!")

In [ ]:
model_key = "mistral-24b"
model_id = MODELS[model_key]["model_id"]
output_dir = f"./output/{model_key}"

# Load model
model, tokenizer = load_model_and_tokenizer(model_id)

# Train
model, tokenizer, metrics, training_time = train_model(model, tokenizer, model_key, output_dir)

# Evaluate
eval_results = evaluate_model(model, tokenizer, model_key)

# Store results
results[model_key] = {
    'training_time': training_time,
    'eval_results': eval_results,
    'metrics': metrics,
    'output_dir': output_dir
}

# Clear memory
del model, tokenizer
torch.cuda.empty_cache()

print(f"\n✓ {model_key} training completed!")

In [ ]:
# Create comparison dataframe
comparison_data = []

for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Training Time (min)': result['training_time'] / 60,
        'Avg Perplexity': result['eval_results']['avg_perplexity'],
        'Test Samples': result['eval_results']['num_test_samples']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Avg Perplexity')

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save comparison to CSV
comparison_df.to_csv('./model_comparison.csv', index=False)
print("\n✓ Comparison saved to model_comparison.csv")

In [ ]:
# Display sample predictions from each model
print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

for model_name, result in results.items():
    print(f"\n{'─'*80}")
    print(f"Model: {model_name}")
    print(f"{'─'*80}")
    
    for i, (prompt, prediction) in enumerate(zip(
        result['eval_results']['sample_prompts'][:3],
        result['eval_results']['sample_predictions'][:3]
    )):
        print(f"\nSample {i+1}:")
        print(f"Prompt: {prompt[:200]}...")
        print(f"Response: {prediction[:300]}...")
        print()

## 7. Upload Models to HuggingFace Hub

In [ ]:
# Upload all fine-tuned models with comprehensive README
upload_results = {}

print("\n" + "="*80)
print("UPLOADING MODELS TO HUGGINGFACE HUB (ahczhg)")
print("="*80 + "\n")

for model_key, model_config in MODELS.items():
    if model_key in results:
        output_dir = results[model_key]['output_dir']
        hub_name = model_config['hub_name']
        
        # Pass model_key as the 4th parameter
        success = upload_to_hub(output_dir, hub_name, model_key, model_key)
        upload_results[model_key] = success
        
        print("\n" + "-"*80 + "\n")

print("\n" + "="*80)
print("UPLOAD SUMMARY")
print("="*80)
for model_key, success in upload_results.items():
    status = "✓ Success" if success else "✗ Failed"
    hub_url = f"https://huggingface.co/{MODELS[model_key]['hub_name']}"
    print(f"{model_key}: {status}")
    if success:
        print(f"  → {hub_url}")
print("="*80)

## 8. Final Summary and Analysis

In [ ]:
# Create final summary report
print("\n" + "="*80)
print("FINAL SUMMARY REPORT")
print("="*80)
print(f"\nDataset: {DATASET_NAME}")
print(f"Number of samples used: {NUM_SAMPLES}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(valid_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nModels fine-tuned: {len(results)}")

print("\n" + "─"*80)
print("BEST PERFORMING MODEL (by Perplexity):")
print("─"*80)
best_model = comparison_df.iloc[0]
print(f"Model: {best_model['Model']}")
print(f"Perplexity: {best_model['Avg Perplexity']:.2f}")
print(f"Training Time: {best_model['Training Time (min)']:.2f} minutes")

print("\n" + "─"*80)
print("FASTEST TRAINING MODEL:")
print("─"*80)
fastest_model = comparison_df.nsmallest(1, 'Training Time (min)').iloc[0]
print(f"Model: {fastest_model['Model']}")
print(f"Training Time: {fastest_model['Training Time (min)']:.2f} minutes")
print(f"Perplexity: {fastest_model['Avg Perplexity']:.2f}")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print("1. All models were successfully fine-tuned using LoRA")
print(f"2. Total training time across all models: {sum(comparison_df['Training Time (min)']):.2f} minutes")
print(f"3. Perplexity range: {comparison_df['Avg Perplexity'].min():.2f} - {comparison_df['Avg Perplexity'].max():.2f}")
print(f"4. All models uploaded to HuggingFace Hub")
print("\n" + "="*80)

# Save full results to JSON
results_summary = {
    'dataset': DATASET_NAME,
    'num_samples': NUM_SAMPLES,
    'lora_config': LORA_CONFIG,
    'training_config': TRAINING_CONFIG,
    'results': {k: {
        'training_time_minutes': v['training_time'] / 60,
        'avg_perplexity': v['eval_results']['avg_perplexity'],
        'hub_name': MODELS[k]['hub_name']
    } for k, v in results.items()}
}

with open('./results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("\n✓ Full results saved to results_summary.json")
print("\n" + "="*80)
print("EXPERIMENT COMPLETED SUCCESSFULLY!")
print("="*80)

## 9. Test Fine-tuned Models (Optional)

In [ ]:
# Interactive testing cell - run this to test any model
def test_model_interactive(model_key, prompt):
    """Test a specific model with a custom prompt"""
    if model_key not in results:
        print(f"Model {model_key} not found in results")
        return
    
    output_dir = results[model_key]['output_dir']
    
    # Load model
    print(f"Loading {model_key}...")
    tokenizer = AutoTokenizer.from_pretrained(output_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        output_dir,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16
    )
    
    # Generate
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7
    )
    
    output = generator(prompt)[0]['generated_text']
    
    print(f"\n{'='*80}")
    print(f"Model: {model_key}")
    print(f"{'='*80}")
    print(f"\nPrompt:\n{prompt}")
    print(f"\nGenerated Response:\n{output}")
    print(f"\n{'='*80}")
    
    # Cleanup
    del model, tokenizer, generator
    torch.cuda.empty_cache()

# Example usage:
# test_model_interactive("qwen3-0.6b", "### Instruction:\nIs this content safe? 'Hello, how are you?'\n\n### Response:\n")

In [ ]:
# This cell manually creates and uploads README.md files to fix missing model information
from huggingface_hub import HfApi, upload_file
import os

def create_and_upload_readme(model_key):
    """Generate and upload README.md for a specific model"""
    
    if model_key not in MODELS:
        print(f"❌ Model key '{model_key}' not found in MODELS configuration")
        return False
    
    model_config = MODELS[model_key]
    hub_name = model_config['hub_name']
    
    # Get training results if available
    training_results = results.get(model_key, None)
    
    # Generate README content
    print(f"\n{'='*80}")
    print(f"Generating README for: {hub_name}")
    print(f"{'='*80}")
    
    readme_content = generate_model_card(model_key, model_config, training_results)
    
    # Save locally first
    temp_readme_path = f"./temp_README_{model_key}.md"
    with open(temp_readme_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)
    
    print(f"✓ README.md generated ({len(readme_content)} characters)")
    print(f"✓ Saved temporarily to: {temp_readme_path}")
    
    # Upload to HuggingFace Hub
    try:
        print(f"\nUploading README.md to {hub_name}...")
        
        api = HfApi()
        api.upload_file(
            path_or_fileobj=temp_readme_path,
            path_in_repo="README.md",
            repo_id=hub_name,
            repo_type="model",
            commit_message="Update README with comprehensive model information"
        )
        
        print(f"✅ Successfully uploaded README.md!")
        print(f"🔗 View at: https://huggingface.co/{hub_name}")
        
        # Clean up temp file
        if os.path.exists(temp_readme_path):
            os.remove(temp_readme_path)
            
        return True
        
    except Exception as e:
        print(f"❌ Error uploading README: {e}")
        print(f"\n📝 README content saved locally at: {temp_readme_path}")
        print(f"   You can manually upload it to: https://huggingface.co/{hub_name}/blob/main/README.md")
        return False


# Upload README for all three models
print("\n" + "="*80)
print("UPDATING README FILES ON HUGGINGFACE HUB")
print("="*80)

upload_status = {}

for model_key in ["qwen3-0.6b", "llama-3.2-1b", "ernie-0.3b"]:
    success = create_and_upload_readme(model_key)
    upload_status[model_key] = success
    print("\n" + "-"*80)

# Summary
print("\n" + "="*80)
print("README UPLOAD SUMMARY")
print("="*80)

for model_key, success in upload_status.items():
    hub_name = MODELS[model_key]['hub_name']
    status = "✅ Success" if success else "❌ Failed"
    print(f"{model_key}: {status}")
    print(f"   → https://huggingface.co/{hub_name}")

print("\n" + "="*80)
print("All README files have been updated!")
print("Please refresh your HuggingFace model pages to see the updated information.")
print("="*80)

## 10. Manually Update README for Models (Fix Missing Information)